In [ ]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression, LassoCV, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error,  accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.datasets import make_regression
from datetime import datetime


covid_df = pd.read_csv(r"C:\Users\Negus\OneDrive\שולחן העבודה\Excel Files\Covid19_With_GDP_Values.csv")
covid_df

In [ ]:
#i decided to remove this columns because there is not much datas in them
covid_df = covid_df.drop(columns = ["Province/State", "Unnamed: 0", "Country/Region"])

covid_df["Recovered"] = covid_df["Confirmed"] - covid_df["Deaths"] 
covid_df

In [ ]:
#check for missing values in the dataset
covid_df.isnull().sum()

In [ ]:
#drop the missing data set 
covid_df = covid_df.dropna()
covid_df

In [ ]:
#check for duplicate rows
is_duplicated = covid_df.duplicated().sum()
is_duplicated

In [ ]:
#add column year instade date column 
covid_df["Year"] = pd.to_datetime(covid_df["Date"]).dt.year
# #Delete Date column 
covid_df = covid_df.drop(columns = "Date")
covid_df



In [ ]:
df = covid_df.groupby(['Year']).mean().reset_index()
df

In [ ]:
# delete the year 
covid_df = covid_df.drop(columns = "Year")

In [ ]:
#correlations with the fetures
correlation_matrix = covid_df.corr()
correlation_matrix

In [ ]:
# Generate a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
#feture corelations with the label
correlations_with_label = correlation_matrix['GDP']

correlations_with_label

In [ ]:
#feture corelations with each other 
correlations_ = correlation_matrix.drop("GDP", axis = 1)
correlations_

In [ ]:
covid_df


In [ ]:
sns.pairplot(covid_df, diag_kind = 'hist')

In [ ]:
X = covid_df.drop('GDP', axis = 1)
Y = covid_df[['GDP']]

scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)

#show the features in dataframe
scaled_X = pd.DataFrame(data = scaled_X, columns = X.columns)
scaled_X

X_train, X_test, y_train, y_test = train_test_split(scaled_X, Y, test_size=0.3, random_state=42)

Linear_model = LinearRegression()
Linear_model.fit(X_train, y_train)

y_pred = Linear_model.predict(X_test)

# ⦁	Perform Vanilla Linear Regression with K-fold cross validation
mse = cross_val_score(Linear_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-mse)

print("MSE:", np.mean(mse))
print("-"*30)
print("RMSE:", np.mean(rmse))
print("-"*30)
print("Coefficiens:")
print(Linear_model.coef_)


In [ ]:
# ⦁	Perform RidgeCV Regression with optimal lambda parameter, show your calculation to find the optimal parameter
alpha_range = np.logspace(-5, 5, 100)

ridge_cv = RidgeCV(alphas=alpha_range, store_cv_values=True)
ridge_cv.fit( X_train, y_train)

#prediction
ridge_pred = ridge_cv.predict(X_test)


mse_ridge = mean_squared_error(y_test, ridge_pred)
rmse_ridge = np.sqrt(mean_squared_error(y_test, ridge_pred))

print(f"MSE Ridge:{mse_ridge}")
print("-"*30)
print(f"RMSE Ridge:{rmse_ridge}")
print("-"*30)

#  the optimal alpha
optimal_alpha_ridge = ridge_cv.alpha_
print(f"Optimal Alpha for Ridge: {optimal_alpha_ridge}")
print("-"*30)
print("Coefficiens:")
print(ridge_cv.coef_)

In [ ]:
# ⦁	Perform LassoCV Régression with optimal lambda parameter, show your calculation to find the optimal parameter, 

lasso_cv = LassoCV(eps=0.1, n_alphas = 100)
lasso_cv.fit( X_train, y_train)

#prediction
lasso_pred = lasso_cv.predict(X_test)


mse_lasso = mean_squared_error(y_test, lasso_pred)
rmse_lasso = np.sqrt(mean_squared_error(y_test, lasso_pred))

print(f"MSE Lasso:{mse_lasso}")
print("-"*30)
print(f"RMSE Lasso:{rmse_lasso}")
print("-"*30)

#  the optimal alpha
optimal_alpha_lasso = lasso_cv.alpha_
print(f"Optimal Alpha  for Lasso: {optimal_alpha_lasso}")
print("-"*30)
print("Coefficiens:")
print(lasso_cv.coef_)

In [ ]:
# ⦁	Perform Polynomial Regression with optimal polynomial degree, show your calculation to find the optimal degree
test_errors_rmse = []
train_errors_rmse = []
for d in range(1, 8): 

    poly_convert = PolynomialFeatures(degree=d, include_bias=False)
    X_convert = poly_convert.fit_transform(scaled_X)
    
    X_train, X_test, y_train, y_test = train_test_split(X_convert, Y, test_size=0.3, random_state=42)

    poly_model = LinearRegression()
    poly_model.fit(X_train, y_train)
    
    test_predictions = poly_model.predict(X_test)
    train_predictions = poly_model.predict(X_train)

    
    poly_mse = mean_squared_error(y_test, test_predictions)
    
    test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
    train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

    
    test_errors_rmse.append(test_rmse)
    train_errors_rmse.append(train_rmse)

print(test_errors_rmse)
print(train_errors_rmse)

plt.plot(range(1, 8), test_errors_rmse, label="Train_RMSE")
plt.plot(range(1, 8), train_errors_rmse, label="Test_RMSE")

plt.ylabel("RMSE")
plt.xlabel("Degree of Polynomial")
plt.legend()
plt.show()



In [ ]:
#the optimal dgree is can be 1-6 i decide to use 4
poly_convert = PolynomialFeatures(degree=4, include_bias=False)
X_convert = poly_convert.fit_transform(scaled_X)

X_train, X_test, y_train, y_test = train_test_split(X_convert, Y, test_size=0.3, random_state=42)

poly_model = LinearRegression()
poly_model.fit(X_train, y_train)

poly_predictions = poly_model.predict(X_test)

poly_mse = mean_squared_error(y_test, poly_predictions)
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_predictions))

print("MSE:", poly_mse)
print("-"*30)
print("RMSE:", poly_rmse)
print("-"*30)
print("Coefficients:")
print( poly_model.coef_)

In [ ]:
models = ['Linear Regression', 'Ridge', 'Lasso', 'Polynomial']
rmse_values = [np.mean(rmse), rmse_ridge, rmse_lasso, poly_rmse]
plt.plot(models, rmse_values)
plt.xlabel('Models')
plt.ylabel('RMSE')
plt.show()

In [ ]:
model_errors = {
    'Linear': np.mean(rmse),
    'Ridge': rmse_ridge,
    'Lasso': rmse_lasso,
    'Polynomial': poly_rmse
}
print(model_errors.values())
best_model = min(model_errors, key = model_errors.get )
print(f"Best model based on RMSE: {best_model}")

In [ ]:
from joblib import dump, load
final_model = LassoCV(alphas = [lasso_cv.alpha_])
final_model.fit(X, Y)

dump(final_model, 'final_model.joblib')
dump(scaler, 'scaler.joblib')

loaded_model = load('final_model.joblib')
loaded_scaler = load('scaler.joblib')

new_data = pd.DataFrame({
    'Confirmed':[1403.0,54876.0,469.0,8117.0,219218.0],
    'Deaths':[149.0,12567.0,132.0,84.0,6284.0],
    'Recovered':[1254.0, 257.0,337.0,8033.0,216734.0],
    'Unemployment':[11.3, 16.34, 24.345,12.437,12.437],
    'CPI':[111.301113,129.302343,105.30113,181.619932,182.619932]
    
})

new_data_scaled = loaded_scaler.transform(new_data)
print(new_data)
print('---------------------')

prediction = loaded_model.predict(new_data_scaled)

print("new_data prediction GDP : ")
print(prediction)